# 99 — Betsy Gun nodal vs Geode single-shot alignment

Purpose:

1. Find the Betsy Gun Geode shot metadata.
2. Find the corresponding **single** nodal event.
3. Plot the Betsy Gun shot gather on the nodals.
4. Plot the Geode gather for comparison.
5. Correlate Geode and nodal data at common receiver positions.
6. Estimate the best overall Geode→nodal time shift.
7. Produce a combined normalized shot gather on one plot.

This notebook is deliberately focused on the Betsy Gun single shot, not the repeated hammer/PEG stacks.

Inputs:
- `geode_events`
- `nodal_source_estimates`
- `shot_gather_files`
- `trace_index`

Outputs/replaces only:
- `betsy_nodal_geode_alignment`
- `betsy_alignment_trace_correlations`
- `betsy_alignment_files`
- `betsy_alignment_errors`

In [3]:
from pathlib import Path
import sys
import sqlite3
import traceback
import importlib.util
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from obspy import read, Stream, UTCDateTime

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CATALOG_DB.parent.mkdir(parents=True, exist_ok=True)

OUT_ROOT = PROJECT_ROOT / "betsy_gun_alignment_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.append('../../lib')
try:
    from segy_tools.gather import stream_to_gather_arrays
    print("Imported segy_tools.gather from package path.")
except Exception as e:
    print("Could not import segy_tools.gather:", repr(e))
    local_gather = Path("/mnt/data/gather.py")
    if local_gather.exists():
        spec = importlib.util.spec_from_file_location("gather_local", local_gather)
        gather_local = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(gather_local)
        stream_to_gather_arrays = gather_local.stream_to_gather_arrays
        print("Imported uploaded gather.py fallback.")
    else:
        raise

COMPONENT = "Z"

# Search/matching controls.
BETSY_SEARCH_TERMS = ["betsy", "betsey", "buffalo"]
MANUAL_GEODE_EVENT_ID = None       # set if automatic Betsy search finds wrong row
MANUAL_NODAL_EVENT_ID = None       # set if automatic nodal match finds wrong event

NODAL_SEARCH_BEFORE_S = 15.0
NODAL_SEARCH_AFTER_S = 15.0
SOURCE_TOLERANCE_M = 8.0

# Correlation controls.
BANDPASS_FREQMIN_HZ = 5.0
BANDPASS_FREQMAX_HZ = 150.0
XCORR_TMIN_S = 0.0
XCORR_TMAX_S = 0.8
MAX_SHIFT_S = 0.25
COMMON_RECEIVER_TOL_M = 1.1
MIN_TRACE_CORR = 0.30

# Plot controls.
PLOT_TMIN_S = 0.0
PLOT_TMAX_S = 0.8
CLIP_PERCENTILE = 99
TRACE_SCALE = 0.7

WRITE_TABLES = True

print("CATALOG_DB:", CATALOG_DB)
print("OUT_ROOT:", OUT_ROOT)

Imported segy_tools.gather from package path.
CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/betsy_gun_alignment_v1


## 1. Load required catalog tables

In [4]:
REQUIRED = ["geode_events", "nodal_source_estimates", "shot_gather_files", "trace_index"]
OWNED = [
    "betsy_nodal_geode_alignment",
    "betsy_alignment_trace_correlations",
    "betsy_alignment_files",
    "betsy_alignment_errors",
]

if not CATALOG_DB.exists():
    raise FileNotFoundError(CATALOG_DB)

with sqlite3.connect(CATALOG_DB) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"].tolist()

missing = [t for t in REQUIRED if t not in tables]
if missing:
    raise RuntimeError(f"Missing required inputs: {missing}. Run 90, 92, and 93 first.")

conn = sqlite3.connect(CATALOG_DB)
geode_events = pd.read_sql("SELECT * FROM geode_events", conn)
nodal_source_estimates = pd.read_sql("SELECT * FROM nodal_source_estimates", conn)
shot_gather_files = pd.read_sql("SELECT * FROM shot_gather_files WHERE instrument_system='nodal'", conn)
trace_index = pd.read_sql("SELECT * FROM trace_index WHERE instrument_system='nodal'", conn)

geode_events["geode_final_trigger_dt"] = pd.to_datetime(geode_events["geode_final_trigger_time_utc"], errors="coerce", utc=True)
geode_events["source_x_m"] = pd.to_numeric(geode_events["source_x_m"], errors="coerce")

nodal_source_estimates["event_time_dt"] = pd.to_datetime(nodal_source_estimates["event_time_utc"], errors="coerce", utc=True)
nodal_source_estimates["estimated_source_x_m"] = pd.to_numeric(nodal_source_estimates["estimated_source_x_m"], errors="coerce")

print("geode_events:", len(geode_events))
print("nodal_source_estimates:", len(nodal_source_estimates))
display(geode_events.groupby(["survey", "source_type"], dropna=False).size().reset_index(name="n").head(50))

geode_events: 242
nodal_source_estimates: 3436


,survey,source_type,n
0,T1_1m_refraction,hammer,46
1,T1_2m_refraction,hammer,44
2,T1_streamer_masw,PEG,83
3,T3_1m_refraction,hammer,39
4,T4_1m_refraction,hammer,30


## 2. Find Betsy Gun Geode shot

In [5]:
def row_text(row):
    fields = [
        "geode_event_id", "survey", "source_type", "comment", "geode_match_note",
        "source_page", "review_status", "geode_file_path"
    ]
    return " ".join(str(row.get(f, "")) for f in fields).lower()

if MANUAL_GEODE_EVENT_ID:
    betsy_geode = geode_events[geode_events["geode_event_id"].astype(str).eq(MANUAL_GEODE_EVENT_ID)].copy()
else:
    mask = geode_events.apply(lambda r: any(term in row_text(r) for term in BETSY_SEARCH_TERMS), axis=1)
    betsy_geode = geode_events[mask].copy()

betsy_geode = betsy_geode.dropna(subset=["geode_final_trigger_dt", "source_x_m"]).copy()
betsy_geode["geode_file_exists"] = betsy_geode["geode_file_path"].apply(lambda p: Path(str(p)).exists() if pd.notna(p) else False)

print("Betsy candidate Geode rows:", len(betsy_geode))
display(
    betsy_geode[
        [
            "geode_event_id", "survey", "line", "file_no", "source_x_m", "source_type",
            "geode_final_trigger_time_utc", "geode_file_path", "geode_file_exists", "comment"
        ]
    ]
)

if len(betsy_geode) == 0:
    raise RuntimeError(
        "No Betsy Geode rows found. Set MANUAL_GEODE_EVENT_ID near the top of the notebook, "
        "or check whether the source_type/comment in geode_events uses a different spelling."
    )

if len(betsy_geode) > 1:
    print("More than one Betsy candidate found; using the first row for now. Set MANUAL_GEODE_EVENT_ID if needed.")

geode_row = betsy_geode.iloc[0]
print("Selected:", geode_row["geode_event_id"], geode_row["survey"], geode_row["source_x_m"])

Betsy candidate Geode rows: 1


,geode_event_id,survey,line,file_no,source_x_m,source_type,geode_final_trigger_time_utc,geode_file_path,geode_file_exists,comment
87,GEODE_T1_2M_REFRACTION_F3088,T1_2m_refraction,T1,3088.0,47.0,hammer,2026-05-18T23:12:42+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,Betsy / extra source test at 47 m; blow count ...


Selected: GEODE_T1_2M_REFRACTION_F3088 T1_2m_refraction 47.0


## 3. Find matching single nodal event

In [6]:
def nodal_mseed_path(event_id):
    rows = shot_gather_files[
        (shot_gather_files["event_id"].astype(str).eq(str(event_id))) &
        (shot_gather_files["file_type"].astype(str).str.lower().eq("mseed"))
    ]
    if rows.empty:
        return None
    p = Path(rows.iloc[0]["file_path"])
    return p if p.exists() else None

if MANUAL_NODAL_EVENT_ID:
    cand = nodal_source_estimates[nodal_source_estimates["event_id"].astype(str).eq(MANUAL_NODAL_EVENT_ID)].copy()
else:
    t0 = geode_row["geode_final_trigger_dt"]
    sx = float(geode_row["source_x_m"])
    cand = nodal_source_estimates.dropna(subset=["event_time_dt", "estimated_source_x_m"]).copy()

    # Prefer same line if available.
    if "line" in cand.columns and pd.notna(geode_row.get("line")):
        same_line = cand[cand["line"].astype(str).eq(str(geode_row["line"]))].copy()
        if len(same_line):
            cand = same_line

    cand = cand[
        (cand["event_time_dt"] >= t0 - pd.to_timedelta(NODAL_SEARCH_BEFORE_S, unit="s")) &
        (cand["event_time_dt"] <= t0 + pd.to_timedelta(NODAL_SEARCH_AFTER_S, unit="s"))
    ].copy()

    cand["source_x_truth_m"] = sx
    cand["source_x_residual_m"] = cand["estimated_source_x_m"] - sx
    cand["abs_source_x_residual_m"] = cand["source_x_residual_m"].abs()
    cand["time_from_geode_s"] = (cand["event_time_dt"] - t0).dt.total_seconds()
    cand["abs_time_from_geode_s"] = cand["time_from_geode_s"].abs()
    cand = cand[cand["abs_source_x_residual_m"] <= SOURCE_TOLERANCE_M].copy()
    cand = cand.sort_values(["abs_time_from_geode_s", "abs_source_x_residual_m"]).copy()

cand["mseed_path"] = cand["event_id"].apply(nodal_mseed_path)
cand["mseed_exists"] = cand["mseed_path"].apply(lambda p: p is not None and Path(p).exists())

print("Candidate nodal events:", len(cand))
display(
    cand[
        [
            "event_id", "line", "timewindow_label", "event_time_utc",
            "estimated_source_x_m", "source_x_residual_m",
            "time_from_geode_s", "mseed_path", "mseed_exists"
        ]
    ].head(20)
)

cand = cand[cand["mseed_exists"]].copy()
if len(cand) == 0:
    raise RuntimeError(
        "No matching nodal event with MiniSEED found. Increase NODAL_SEARCH_BEFORE/AFTER_S, "
        "SOURCE_TOLERANCE_M, or set MANUAL_NODAL_EVENT_ID."
    )

nodal_row = cand.iloc[0]
print("Selected nodal event:", nodal_row["event_id"], nodal_row["event_time_utc"], nodal_row["estimated_source_x_m"])

Candidate nodal events: 0


,event_id,line,timewindow_label,event_time_utc,estimated_source_x_m,source_x_residual_m,time_from_geode_s,mseed_path,mseed_exists


RuntimeError: No matching nodal event with MiniSEED found. Increase NODAL_SEARCH_BEFORE/AFTER_S, SOURCE_TOLERANCE_M, or set MANUAL_NODAL_EVENT_ID.

In [7]:
import sqlite3
import pandas as pd
from pathlib import Path

CATALOG_DB = Path("/Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite")
conn = sqlite3.connect(CATALOG_DB)

t0 = pd.Timestamp("2026-05-18T23:12:42Z")
before_s = 10
after_s = 10

shot_events = pd.read_sql("""
SELECT
    event_id,
    line,
    survey,
    timewindow_label,
    event_time_utc,
    detection_time_utc,
    on_time_utc,
    off_time_utc,
    source_type,
    source_x_m,
    n_receivers_extracted,
    n_traces_extracted,
    status
FROM shot_events
WHERE instrument_system='nodal'
""", conn)

for c in ["event_time_utc", "detection_time_utc", "on_time_utc", "off_time_utc"]:
    shot_events[c + "_dt"] = pd.to_datetime(shot_events[c], errors="coerce", utc=True)

# use whichever time column is populated
shot_events["best_time_dt"] = shot_events["event_time_utc_dt"]
shot_events["best_time_dt"] = shot_events["best_time_dt"].fillna(shot_events["detection_time_utc_dt"])
shot_events["best_time_dt"] = shot_events["best_time_dt"].fillna(shot_events["on_time_utc_dt"])

near = shot_events[
    (shot_events["best_time_dt"] >= t0 - pd.Timedelta(seconds=before_s)) &
    (shot_events["best_time_dt"] <= t0 + pd.Timedelta(seconds=after_s))
].copy()

near["dt_from_betsy_s"] = (near["best_time_dt"] - t0).dt.total_seconds()
display(near.sort_values("dt_from_betsy_s"))

DatabaseError: Execution failed on sql '
SELECT
    event_id,
    line,
    survey,
    timewindow_label,
    event_time_utc,
    detection_time_utc,
    on_time_utc,
    off_time_utc,
    source_type,
    source_x_m,
    n_receivers_extracted,
    n_traces_extracted,
    status
FROM shot_events
WHERE instrument_system='nodal'
': no such column: event_time_utc

In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(CATALOG_DB)

shot_events_cols = pd.read_sql("""
PRAGMA table_info(shot_events)
""", conn)

print(shot_events_cols)

    cid                          name     type  notnull dflt_value  pk
0     0                      event_id     TEXT        0       None   1
1     1             instrument_system     TEXT        0       None   0
2     2                          line     TEXT        0       None   0
3     3                      transect     TEXT        0       None   0
4     4                        survey     TEXT        0       None   0
5     5                   survey_type     TEXT        0       None   0
6     6                       shot_no  INTEGER        0       None   0
7     7                       file_no  INTEGER        0       None   0
8     8                    source_x_m     REAL        0       None   0
9     9                   source_type     TEXT        0       None   0
10   10                       n_blows  INTEGER        0       None   0
11   11                       n_shots  INTEGER        0       None   0
12   12                      operator     TEXT        0       None   0
13   1

In [12]:
import sqlite3
import pandas as pd
from pathlib import Path

CATALOG_DB = Path("/Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite")
conn = sqlite3.connect(CATALOG_DB)

t0 = pd.Timestamp("2026-05-18T23:12:42Z")

shot_events = pd.read_sql("""
SELECT *
FROM shot_events
WHERE instrument_system='nodal'
""", conn)

for c in ["shot_time_utc", "detection_time_utc", "on_time_utc", "off_time_utc"]:
    shot_events[c + "_dt"] = pd.to_datetime(shot_events[c], errors="coerce", utc=True)

shot_events["best_time_dt"] = shot_events["detection_time_utc_dt"]
shot_events["best_time_dt"] = shot_events["best_time_dt"].fillna(shot_events["on_time_utc_dt"])
shot_events["best_time_dt"] = shot_events["best_time_dt"].fillna(shot_events["shot_time_utc_dt"])

near = shot_events[
    shot_events["best_time_dt"].between(
        t0 - pd.Timedelta(seconds=5),
        t0 + pd.Timedelta(seconds=5),
    )
].copy()

near["dt_from_betsy_s"] = (near["best_time_dt"] - t0).dt.total_seconds()

print(
    near.sort_values("dt_from_betsy_s")[
        [
            "event_id",
            "line",
            "survey",
            "timewindow_label",
            "detection_time_utc",
            "on_time_utc",
            "off_time_utc",
            "source_x_m",
            "n_receivers_extracted",
            "n_traces_extracted",
            "status",
            "dt_from_betsy_s",
        ]
    ]
)

                             event_id line            survey  \
2787  T1_N2_Refraction2m_T1_N2_E00449   T1  T1_2m_refraction   

        timewindow_label           detection_time_utc  \
2787  T1_N2_Refraction2m  2026-05-18T23:12:41.934000Z   

                      on_time_utc                 off_time_utc  source_x_m  \
2787  2026-05-18T23:12:41.934000Z  2026-05-18T23:12:42.334000Z        47.0   

      n_receivers_extracted  n_traces_extracted status  dt_from_betsy_s  
2787                     34                 102     ok           -0.066  


In [14]:
shot_gather_files = pd.read_sql("""
SELECT event_id, file_type, file_path
FROM shot_gather_files
WHERE instrument_system='nodal'
""", conn)

near_files = near[["event_id", "dt_from_betsy_s", "timewindow_label"]].merge(
    shot_gather_files,
    on="event_id",
    how="left",
)

near_files["exists"] = near_files["file_path"].apply(
    lambda p: Path(str(p)).exists() if pd.notna(p) else False
)

print(near_files.sort_values(["dt_from_betsy_s", "file_type"]))

                          event_id  dt_from_betsy_s    timewindow_label  \
0  T1_N2_Refraction2m_T1_N2_E00449           -0.066  T1_N2_Refraction2m   

  file_type                                          file_path  exists  
0     mseed  /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_sho...    True  


In [16]:
geode_3088 = pd.read_sql("""
SELECT *
FROM geode_events
WHERE file_no = 3088
""", conn)

print(
    geode_3088[
        [
            "geode_event_id",
            "survey",
            "line",
            "file_no",
            "source_x_m",
            "source_type",
            "geode_final_trigger_time_utc",
            "geode_file_path",
        ]
    ]
)

                 geode_event_id            survey line  file_no  source_x_m  \
0  GEODE_T1_2M_REFRACTION_F3088  T1_2m_refraction   T1   3088.0        47.0   

  source_type geode_final_trigger_time_utc  \
0      hammer    2026-05-18T23:12:42+00:00   

                                     geode_file_path  
0  /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...  


## 4. Geometry and waveform helpers

In [ ]:
def select_component(st, component=COMPONENT):
    stc = st.select(channel=f"*{component}").copy()
    return stc if len(stc) else st.copy()


def infer_geode_receiver_xs(geode_row, n_traces):
    first = pd.to_numeric(geode_row.get("receiver_first_m"), errors="coerce")
    last = pd.to_numeric(geode_row.get("receiver_last_m"), errors="coerce")
    spacing = pd.to_numeric(geode_row.get("receiver_spacing_m"), errors="coerce")

    if np.isfinite(first) and np.isfinite(spacing) and spacing != 0:
        return (first + np.arange(n_traces) * spacing).astype(float), "receiver_first_m + receiver_spacing_m"

    if np.isfinite(first) and np.isfinite(last) and n_traces > 1:
        return np.linspace(first, last, n_traces).astype(float), "linspace(receiver_first_m, receiver_last_m)"

    return np.arange(n_traces, dtype=float), "trace_index_fallback"


def attach_geode_geometry(st, geode_row):
    st = st.copy()
    xs, method = infer_geode_receiver_xs(geode_row, len(st))
    source_x = pd.to_numeric(geode_row.get("source_x_m"), errors="coerce")
    for tr, x in zip(st, xs):
        tr.stats.receiver_x_m = float(x)
        if np.isfinite(source_x):
            tr.stats.source_x_m = float(source_x)
    return st, method


def attach_nodal_geometry(st, event_id, source_x_m=None, component=COMPONENT):
    geom = trace_index[trace_index["event_id"].astype(str).eq(str(event_id))].copy()
    if component:
        geom = geom[geom["channel"].astype(str).str.endswith(component)].copy()

    geom["receiver_x_m"] = pd.to_numeric(geom["receiver_x_m"], errors="coerce")
    geom = geom.dropna(subset=["receiver_x_m"])

    lookup = {}
    for _, r in geom.iterrows():
        station = str(r.get("station", ""))
        channel = str(r.get("channel", ""))
        network = str(r.get("network", ""))
        location = "" if pd.isna(r.get("location", "")) else str(r.get("location", ""))
        lookup[(station, channel)] = float(r["receiver_x_m"])
        lookup[f"{network}.{station}.{location}.{channel}"] = float(r["receiver_x_m"])
        if "seed_id" in geom.columns and pd.notna(r.get("seed_id")):
            lookup[str(r["seed_id"])] = float(r["receiver_x_m"])

    st = st.copy()
    n_attached = 0
    for tr in st:
        x = lookup.get((str(tr.stats.station), str(tr.stats.channel)), lookup.get(tr.id))
        if x is None:
            try:
                x = float(str(tr.stats.station)) / 100.0
            except Exception:
                x = None

        if x is not None and np.isfinite(float(x)):
            tr.stats.receiver_x_m = float(x)
            n_attached += 1

        if source_x_m is not None and np.isfinite(source_x_m):
            tr.stats.source_x_m = float(source_x_m)

    return st, f"trace_index event_id={event_id}; attached {n_attached}/{len(st)} traces"


def shift_stream_to_relative_time(st):
    if len(st) == 0:
        return st
    origin = min(tr.stats.starttime for tr in st)
    out = st.copy()
    for tr in out:
        rel = tr.stats.starttime - origin
        tr.stats.starttime = UTCDateTime(0) + rel
    return out


def preprocess_for_alignment(st):
    st = st.copy()
    for tr in st:
        tr.data = tr.data.astype(np.float64)
        tr.detrend("linear")
        tr.taper(max_percentage=0.02)
        nyq = 0.5 / tr.stats.delta
        if BANDPASS_FREQMAX_HZ < 0.95 * nyq:
            tr.filter("bandpass", freqmin=BANDPASS_FREQMIN_HZ, freqmax=BANDPASS_FREQMAX_HZ, corners=4, zerophase=True)
        else:
            tr.filter("highpass", freq=BANDPASS_FREQMIN_HZ, corners=4, zerophase=True)
    return st


def safe_name(s):
    s = str(s)
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]"]:
        s = s.replace(ch, "_")
    return s

## 5. Load Betsy nodal and Geode gathers

In [ ]:
source_x = float(geode_row["source_x_m"])

st_nodal = read(str(nodal_row["mseed_path"]))
st_nodal = select_component(st_nodal, COMPONENT)
st_nodal, nodal_geom_method = attach_nodal_geometry(st_nodal, nodal_row["event_id"], source_x_m=source_x, component=COMPONENT)
st_nodal = shift_stream_to_relative_time(st_nodal)
st_nodal = preprocess_for_alignment(st_nodal)

st_geode = read(str(geode_row["geode_file_path"]))
st_geode = select_component(st_geode, COMPONENT)
st_geode, geode_geom_method = attach_geode_geometry(st_geode, geode_row)
st_geode = shift_stream_to_relative_time(st_geode)
st_geode = preprocess_for_alignment(st_geode)

print("Nodal:", len(st_nodal), nodal_geom_method)
print("Geode:", len(st_geode), geode_geom_method)
print("Nodal file:", nodal_row["mseed_path"])
print("Geode file:", geode_row["geode_file_path"])

## 6. Convert to gather arrays with `segy_tools`

In [ ]:
t_n, d_n, x_n, sx_n, geom_n = stream_to_gather_arrays(
    st_nodal,
    sort_by="receiver_x",
    component=COMPONENT,
    fallback_source_x_m=source_x,
)

t_g, d_g, x_g, sx_g, geom_g = stream_to_gather_arrays(
    st_geode,
    sort_by="receiver_x",
    component=COMPONENT,
    fallback_source_x_m=source_x,
)

print("Nodal gather:", d_n.shape, "x range", np.nanmin(x_n), np.nanmax(x_n))
print("Geode gather:", d_g.shape, "x range", np.nanmin(x_g), np.nanmax(x_g))

## 7. Find common receivers and estimate Geode→nodal time shift

In [ ]:
def interp_at_time(t_src, y_src, t_grid):
    return np.interp(t_grid, t_src, y_src, left=np.nan, right=np.nan)

def normalize(y):
    y = np.asarray(y, dtype=float)
    y = y - np.nanmedian(y)
    sd = np.nanstd(y)
    if not np.isfinite(sd) or sd <= 0:
        return None
    y[~np.isfinite(y)] = 0
    return y / sd

def xcorr_shift(ref, cand, dt, max_shift_s):
    ref = normalize(ref)
    cand = normalize(cand)
    if ref is None or cand is None:
        return np.nan, np.nan

    cc = correlate(cand, ref, mode="full")
    lags = correlation_lags(len(cand), len(ref), mode="full")
    maxlag = int(round(max_shift_s / dt))
    ok = np.abs(lags) <= maxlag
    if not ok.any():
        return np.nan, np.nan
    cc2 = cc[ok]
    l2 = lags[ok]
    k = int(np.nanargmax(cc2))
    lag = int(l2[k])

    # correlate(candidate, reference): positive lag means candidate is later.
    # shift_to_apply_to_candidate = -lag * dt
    shift_s = -lag * dt

    denom = np.sqrt(np.nansum(ref**2) * np.nansum(cand**2))
    corrcoef = float(cc2[k] / denom) if denom > 0 else np.nan
    return float(shift_s), corrcoef

# Common time grid for correlation.
dt = max(np.nanmedian(np.diff(t_n)), np.nanmedian(np.diff(t_g)))
t_grid = np.arange(XCORR_TMIN_S, XCORR_TMAX_S + 0.5 * dt, dt)

rows = []

for inod, xn in enumerate(x_n):
    j = int(np.nanargmin(np.abs(x_g - xn)))
    dx = float(x_g[j] - xn)
    if abs(dx) > COMMON_RECEIVER_TOL_M:
        continue

    yn = interp_at_time(t_n, d_n[inod], t_grid)
    yg = interp_at_time(t_g, d_g[j], t_grid)

    shift_s, corrcoef = xcorr_shift(yn, yg, dt, MAX_SHIFT_S)

    rows.append({
        "nodal_receiver_x_m": float(xn),
        "geode_receiver_x_m": float(x_g[j]),
        "receiver_dx_m": dx,
        "nodal_trace_index": int(inod),
        "geode_trace_index": int(j),
        "geode_shift_to_apply_s": shift_s,
        "corrcoef": corrcoef,
    })

trace_corrs = pd.DataFrame(rows)
trace_corrs["accepted"] = (
    np.isfinite(trace_corrs["geode_shift_to_apply_s"]) &
    np.isfinite(trace_corrs["corrcoef"]) &
    (trace_corrs["corrcoef"] >= MIN_TRACE_CORR)
)

display(trace_corrs.sort_values("corrcoef", ascending=False).head(20))
print("Common receivers:", len(trace_corrs))
print("Accepted correlations:", int(trace_corrs["accepted"].sum()) if len(trace_corrs) else 0)

if len(trace_corrs) and trace_corrs["accepted"].any():
    best_shift_s = float(trace_corrs.loc[trace_corrs["accepted"], "geode_shift_to_apply_s"].median())
    median_corr = float(trace_corrs.loc[trace_corrs["accepted"], "corrcoef"].median())
else:
    best_shift_s = 0.0
    median_corr = np.nan

print("Best overall Geode shift to apply (s):", best_shift_s)
print("Median accepted corrcoef:", median_corr)

## 8. Combined normalized shot-gather plot

In [ ]:
def normalize_traces_for_plot(data):
    out = data.astype(float).copy()
    out = out - np.nanmedian(out, axis=1)[:, None]
    scale = np.nanpercentile(np.abs(out), CLIP_PERCENTILE, axis=1)
    scale[~np.isfinite(scale) | (scale <= 0)] = 1.0
    return out / scale[:, None]

def plot_single_dataset(ax, time, data, rx, label, linestyle="-", alpha=0.85, shift_s=0.0, color=None):
    mask = (time + shift_s >= PLOT_TMIN_S) & (time + shift_s <= PLOT_TMAX_S)
    tt = time[mask] + shift_s
    dd = normalize_traces_for_plot(data[:, mask])
    rx = np.asarray(rx, dtype=float)

    ux = np.sort(np.unique(rx[np.isfinite(rx)]))
    dx = np.nanmedian(np.diff(ux)) if len(ux) > 1 else 1.0
    if not np.isfinite(dx) or dx <= 0:
        dx = 1.0

    for i, x in enumerate(rx):
        yy = x + TRACE_SCALE * dx * np.clip(dd[i], -1, 1)
        ax.plot(yy, tt, linestyle=linestyle, alpha=alpha, linewidth=0.6, color=color)

fig, ax = plt.subplots(figsize=(14, 8))

plot_single_dataset(ax, t_n, d_n, x_n, label="Nodal", linestyle="-", alpha=0.9, shift_s=0.0, color="black")
plot_single_dataset(ax, t_g, d_g, x_g, label="Geode shifted", linestyle="-", alpha=0.55, shift_s=best_shift_s, color="tab:red")

ax.axvline(source_x, linestyle="--", linewidth=1.2, color="tab:blue", label=f"source x={source_x:.1f} m")
ax.invert_yaxis()
ax.set_xlabel("Receiver x (m)")
ax.set_ylabel("Time since respective trigger/origin (s)")
ax.set_title(
    f"Betsy Gun single shot: nodal + Geode combined normalized gather\n"
    f"Geode shift applied: {best_shift_s:+.4f} s | median corr={median_corr:.2f} | source x={source_x:.1f} m"
)
ax.grid(True, alpha=0.25)
ax.legend()

out_png = OUT_ROOT / f"BETSY_{safe_name(str(geode_row['geode_event_id']))}_{safe_name(str(nodal_row['event_id']))}_combined_shifted.png"
fig.savefig(out_png, dpi=180)
plt.show()

print(out_png)

## 9. Also save separate nodal and Geode panels

In [ ]:
def plot_wiggle_panel(time, data, rx, title, out_png, source_x=None, time_shift_s=0.0):
    fig, ax = plt.subplots(figsize=(13, 7))
    plot_single_dataset(ax, time, data, rx, label=title, shift_s=time_shift_s, color="black")
    if source_x is not None and np.isfinite(source_x):
        ax.axvline(source_x, linestyle="--", linewidth=1.2, color="tab:blue")
    ax.invert_yaxis()
    ax.set_xlabel("Receiver x (m)")
    ax.set_ylabel("Time (s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

out_nodal_png = OUT_ROOT / f"BETSY_{safe_name(str(nodal_row['event_id']))}_nodal_only.png"
out_geode_png = OUT_ROOT / f"BETSY_{safe_name(str(geode_row['geode_event_id']))}_geode_shifted_only.png"

plot_wiggle_panel(t_n, d_n, x_n, "Betsy Gun nodal single-shot gather", out_nodal_png, source_x=source_x, time_shift_s=0.0)
plot_wiggle_panel(t_g, d_g, x_g, f"Betsy Gun Geode gather shifted by {best_shift_s:+.4f} s", out_geode_png, source_x=source_x, time_shift_s=best_shift_s)

print(out_nodal_png)
print(out_geode_png)

## 10. Write notebook-owned tables and CSV exports

In [ ]:
alignment = pd.DataFrame([{
    "geode_event_id": geode_row["geode_event_id"],
    "nodal_event_id": nodal_row["event_id"],
    "survey": geode_row.get("survey"),
    "line": geode_row.get("line"),
    "source_x_m": source_x,
    "geode_final_trigger_time_utc": geode_row.get("geode_final_trigger_time_utc"),
    "nodal_event_time_utc": nodal_row.get("event_time_utc"),
    "nodal_estimated_source_x_m": nodal_row.get("estimated_source_x_m"),
    "nodal_source_x_residual_m": nodal_row.get("source_x_residual_m"),
    "best_geode_shift_to_apply_s": best_shift_s,
    "median_accepted_corrcoef": median_corr,
    "n_common_receivers": len(trace_corrs),
    "n_accepted_trace_correlations": int(trace_corrs["accepted"].sum()) if len(trace_corrs) else 0,
    "nodal_file_path": str(nodal_row["mseed_path"]),
    "geode_file_path": str(geode_row["geode_file_path"]),
}])

files = pd.DataFrame([
    {"file_type": "png_combined_shifted", "file_path": str(out_png)},
    {"file_type": "png_nodal_only", "file_path": str(out_nodal_png)},
    {"file_type": "png_geode_shifted_only", "file_path": str(out_geode_png)},
])

errors = pd.DataFrame(columns=["stage", "error"])

alignment.to_csv(OUT_ROOT / "betsy_nodal_geode_alignment.csv", index=False)
trace_corrs.to_csv(OUT_ROOT / "betsy_alignment_trace_correlations.csv", index=False)
files.to_csv(OUT_ROOT / "betsy_alignment_files.csv", index=False)

if WRITE_TABLES:
    with sqlite3.connect(CATALOG_DB) as conn:
        for t in OWNED:
            conn.execute(f'DROP TABLE IF EXISTS "{t}"')
        alignment.to_sql("betsy_nodal_geode_alignment", conn, if_exists="fail", index=False)
        trace_corrs.to_sql("betsy_alignment_trace_correlations", conn, if_exists="fail", index=False)
        files.to_sql("betsy_alignment_files", conn, if_exists="fail", index=False)
        errors.to_sql("betsy_alignment_errors", conn, if_exists="fail", index=False)
        conn.commit()

display(alignment)
display(files)
print("Wrote outputs to:", OUT_ROOT)